# 09 - Đo lường hiệu năng thời gian thực & Khả năng mở rộng Spark

### Khoảng trống nghiên cứu giải quyết (Research Gap):
- Bài báo gốc trình bày khả năng mở rộng (scalability) và hiệu năng xử lý dữ liệu lớn bằng cụm Apache Spark 5 nodes tại Table 8-9 (đạt throughput 98,500+ events/sec).
- Notebook này tái hiện và **đo lường trực tiếp hiệu năng tiền xử lý và suy luận** trên môi trường cục bộ để xác nhận tốc độ thực thi thực tế, đồng thời theo dõi mức độ chiếm dụng tài nguyên hệ thống (CPU, RAM, GPU) của pipeline.

In [1]:
# === Thiết lập môi trường ỔN ĐỊNH + CHỐNG RỚT DRIVE (auto-retry remount) ===
# Làm việc trên Ổ CỨNG LOCAL /content; kéo dữ liệu từ Drive có thử lại khi mount rớt.
import os, time, shutil
from contextlib import contextmanager

DRIVE = '/content/drive/MyDrive/Nhom28_CyberDetect_MLP_Final'
PROJECT_PATH = '/content/cdmlp'
NEED_RAW = False   # True chi voi notebook 08
ON_COLAB = False

def _mount():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

try:
    _mount(); ON_COLAB = True
except Exception:
    here = os.getcwd()
    while here != os.path.dirname(here) and not os.path.isdir(os.path.join(here, 'notebooks')):
        here = os.path.dirname(here)
    DRIVE = here; PROJECT_PATH = here

def _pull(rel):
    s = os.path.join(DRIVE, rel); d = os.path.join(PROJECT_PATH, rel)
    if os.path.isdir(s): shutil.copytree(s, d, dirs_exist_ok=True)
    elif os.path.isfile(s):
        os.makedirs(os.path.dirname(d), exist_ok=True); shutil.copy2(s, d)

if ON_COLAB:
    os.makedirs(PROJECT_PATH, exist_ok=True)
    rels = ['scripts', 'src', 'models', 'results', 'data/colab_processed', 'data/ton_iot.csv']
    if NEED_RAW: rels.append('data/raw')
    # ton_iot.csv là item CUỐI -> nếu nó có mặt tức là đã kéo xong toàn bộ list
    sentinel = os.path.join(PROJECT_PATH, 'data', 'ton_iot.csv')
    ok = False
    for attempt in range(1, 6):
        for rel in rels:
            try: _pull(rel)
            except Exception as e: print(f'  (loi pull {rel}, thu lai sau): {str(e)[:70]}')
        if os.path.exists(sentinel):
            ok = True; break
        print(f'  [retry {attempt}/5] Drive rot khi keo du lieu -> remount + thu lai...')
        try: _mount()
        except Exception: pass
        time.sleep(3)
    if not ok:
        print('  [CANH BAO] Drive khong on dinh. Hay Runtime > Restart session roi chay lai cell nay.')
    else:
        print('  [OK] Da keo du lieu tu Drive ve local.')

os.makedirs(PROJECT_PATH, exist_ok=True); os.chdir(PROJECT_PATH)
try:
    get_ipython().run_line_magic('cd', PROJECT_PATH)
except Exception:
    pass
for d in ('data/colab_processed', 'results', 'results/xai', 'models'):
    os.makedirs(d, exist_ok=True)
print('PROJECT_PATH (local) =', PROJECT_PATH, '| ON_COLAB =', ON_COLAB)

def find_data_csv():
    for c in ['data/ton_iot.csv', 'data/raw/ton_iot.csv', 'ton_iot.csv']:
        if os.path.exists(c): return c
    raise FileNotFoundError('Khong tim thay ton_iot.csv (Drive: data/ton_iot.csv).')

@contextmanager
def step(name):
    t0=time.time(); print(f'\n[START] {name}', flush=True)
    try: yield
    finally: print(f'[DONE] {name} - {time.time()-t0:.1f}s', flush=True)

def sync_to_drive(retries=5):
    if not ON_COLAB:
        print('[local] bo qua sync Drive'); return
    for attempt in range(1, retries + 1):
        try:
            for rel in ['data/colab_processed', 'models', 'results']:
                s = os.path.join(PROJECT_PATH, rel)
                if os.path.exists(s): shutil.copytree(s, os.path.join(DRIVE, rel), dirs_exist_ok=True)
            print('[OK] Da dong bo ket qua ve Drive:', DRIVE); return
        except Exception as e:
            print(f'  [retry sync {attempt}/{retries}] Drive rot -> remount: {str(e)[:70]}')
            try: _mount()
            except Exception: pass
            time.sleep(3)
    print('  [CANH BAO] Sync Drive that bai. Ket qua van o local /content/cdmlp.')


Mounted at /content/drive
  [OK] Da keo du lieu tu Drive ve local.
/content
PROJECT_PATH (local) = /content/cdmlp | ON_COLAB = True


In [2]:
# Cài đặt các thư viện bổ trợ cần thiết
!pip -q install imbalanced-learn pyarrow joblib psutil tqdm scipy


In [3]:
# Thêm project root vào system path để import mô hình và modules
import sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

import numpy as np
import pandas as pd

## 1. Đo lường tốc độ tiền xử lý dữ liệu của Spark

Chúng ta sẽ thực thi kịch bản đo độ trễ tiền xử lý Spark và độ trễ suy luận của mô hình MLP trên từng gói tin để đánh giá throughput (số lượng sự kiện xử lý được trên mỗi giây):

In [4]:
# Chạy benchmark đo lường độ trễ và throughput
!python scripts/benchmark_latency.py

2026-06-03 16:11:21.795383: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-03 16:11:27.109433: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1780503087.110864    5180 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
[INFO] Test samples: (42209, 30)
2026-06-03 16:11:28.938263: I external/local_xla/xla/service/service.cc:163] XLA service 0x7f57540048c0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-06-03 16:11:2

## 2. Đo lường khả năng mở rộng và giám sát tài nguyên phần cứng

Tiếp theo, chúng ta chạy kịch bản đo lường khả năng mở rộng khi kích thước dữ liệu mạng tăng dần từ 10% đến 100% dung lượng tập dữ liệu, đồng thời giám sát CPU và Memory sử dụng thực tế:

In [5]:
# Chạy benchmark khả năng mở rộng với kích thước dữ liệu tăng dần
!python scripts/benchmark_scalability.py

2026-06-03 16:11:41.339844: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-06-03 16:11:45.576546: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1780503105.577970    5547 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
2026-06-03 16:11:46.720353: I external/local_xla/xla/service/service.cc:163] XLA service 0x7c8cfc0038b0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-06-03 16:11:46.720382: I external/local_xla/xl

### Nhận xét & Đóng góp học thuật thực tế:
1. **Độ trễ suy luận đáp ứng thời gian thực:** Kết quả thực nghiệm cho thấy thời gian suy luận của mô hình MLP cực kỳ nhanh, thấp nhất khoảng **0.053 ms/mẫu** ở chế độ batch lớn nhất, đạt throughput tối đa **~18,810 sự kiện/giây** (Bảng 8 - đo thật) trên tài nguyên đơn máy. Điều này đáp ứng hoàn hảo yêu cầu bảo mật giám sát thời gian thực của thiết bị IoT.
2. **Tính khả thi của Apache Spark:** Tốc độ của Spark trong việc chuẩn hóa và làm sạch lượng dữ liệu lớn trước khi đưa vào mô hình là động lực cốt lõi giúp hệ thống có thể mở rộng tuyến tính khi triển khai trên cụm máy chủ phân tán thực tế.

In [7]:
# === ĐỒNG BỘ KẾT QUẢ VỀ GOOGLE DRIVE (chạy cuối cùng, sau khi các cell trên xong) ===
sync_to_drive()


[OK] Da dong bo ket qua ve Drive: /content/drive/MyDrive/Nhom28_CyberDetect_MLP_Final
